In [ ]:
import os
import time 
import numpy as np
import pandas as pd
import akshare as ak
import quantstats as qs
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

import akshare_proxy_patch

akshare_proxy_patch.install_patch(
    "101.201.173.125",
    # 免费版为空，付费版填入具体TOKEN
    auth_token="",
    retry=30,
    # 封控的域名列表，可动态调整
    hook_domains=[
      "fund.eastmoney.com",
      "push2.eastmoney.com",
      "push2his.eastmoney.com",
      "emweb.securities.eastmoney.com",
    ],
)

In [ ]:
# 510300：沪深300ETF，代表大盘
# 510500：中证500ETF，代表小盘
# 510880：红利ETF，代表价值
# 159915：创业板ETF，代表成长
# 513100：纳指ETF，代表外盘
# 518880：黄金ETF，代表商品
code_list = ['510300', '510500', '510880', '159915', '513100', '518880']
start_date = '20150101'
end_date = '20260301'

df_list = []
for code in code_list:
    print(f'正在获取[{code}]行情数据...')
    # adjust：""-不复权、qfq-（前复权）、hfq-后复权
    df = ak.fund_etf_hist_em(symbol=code, period='daily', 
        start_date=start_date, end_date=end_date, adjust='hfq')
    df.insert(0, 'code', code)
    df_list.append(df)
    time.sleep(3)
print('数据获取完毕！')

# ETF名称
code_to_name_dict = {
    '510300': '沪深300ETF华泰柏瑞',
    '510500': '中证500ETF',
    '510880': '红利ETF华泰柏瑞',
    '159915': '创业板ETF易方达',
    '513100': '纳指ETF',
    '518880': '黄金ETF华安'
}

all_df = pd.concat(df_list, ignore_index=True)
data = all_df.pivot(index='日期', columns='code', values='收盘')[code_list]
data.index = pd.to_datetime(data.index)
data = data.sort_index()
data.head(10)

In [ ]:
# # 纯动量轮动
# # 动量长度
# N = 10
# # 计算每日涨跌幅和N日涨跌幅
# for code in code_list:
#     data['日收益率_'+code] = data[code] / data[code].shift(1) - 1.0
#     data['涨幅_'+code] = data[code] / data[code].shift(N+1) - 1.0
# # 去掉缺失值
# data = data.dropna()
# data[['涨幅_'+v for v in code_list]].head(10)



# # RSRS 阻力支撑强弱指标
# # 计算强弱得分
# def calculate_score(srs, N=25):
#     if srs.shape[0] < N:
#         return np.nan
#     x = np.arange(1, N+1)
#     y = srs.values / srs.values[0]
#     lr = LinearRegression().fit(x.reshape(-1, 1), y)
#     # 斜率
#     slope = lr.coef_[0]
#     # 决定系数R2
#     r_squared = lr.score(x.reshape(-1, 1), y)
#     # 得分
#     score = 10000 * slope * r_squared
#     return score

# # 斜率计算长度
# N = 25
# # 计算每日涨跌幅和得分
# for code in code_list:
#     data['日收益率_'+code] = data[code] / data[code].shift(1) - 1.0
#     data['得分_'+code] = data[code].rolling(N).apply(lambda x: calculate_score(x, N))
# # 去掉缺失值
# data = data.dropna()
# data[['得分_'+v for v in code_list]].head(10)



# ATR波动率指标
# ================= 1. 参数设置 =================
lb_min = 10          # 最小回溯窗口 (短期)
lb_max = 60          # 最大回溯窗口 (长期)
vol_short_len = 10   # 用于计算短期波动率的窗口
vol_long_len = 60    # 用于计算长期波动率的窗口
ratio_cap = 0.9      # 比例阈值上限

# ================= 2. 改造得分计算函数 =================
# 因为采用动态窗口，传入的不再是固定的 pandas Series，而是动态长度的 numpy array
def calculate_score_dynamic(prices):
    if pd.isna(prices).any() or prices[0] == 0:
        return np.nan
    
    N = len(prices)
    if N < 2:
        return np.nan
    x = np.arange(1, N + 1).reshape(-1, 1)
    y = prices / prices[0] # 归一化
    
    lr = LinearRegression().fit(x, y)
    slope = lr.coef_[0]
    r_squared = lr.score(x, y)
    
    score = 10000 * slope * r_squared
    return score

# ================= 3. 计算波动率与动态窗口 =================
for code in code_list:
    # 计算日收益率
    data['日收益率_'+code] = data[code] / data[code].shift(1) - 1.0
    
    # 计算长短期的历史波动率 (收益率的标准差)
    vol_short = data['日收益率_'+code].rolling(vol_short_len).std()
    vol_long = data['日收益率_'+code].rolling(vol_long_len).std()
    
    # 计算 vol_ratio 并用 ratio_cap 进行截断
    vol_ratio = vol_short / vol_long
    vol_ratio_capped = vol_ratio.clip(upper=ratio_cap)
    
    # 核心公式：计算动态回溯窗口 lookback
    lookback = lb_min + (lb_max - lb_min) * (1 - vol_ratio_capped)
    
    # 将窗口天数向下取整，转为整数。数据不足的部分默认使用最大窗口 lb_max
    data['lookback_'+code] = np.floor(lookback).fillna(lb_max).astype(int)

# ================= 4. 计算动态动量得分 =================
for code in code_list:
    # 预生成一个全是 NaN 的数组
    scores = np.full(len(data), np.nan)
    
    # 从 lb_max 开始遍历，确保有足够的历史数据
    for i in range(lb_max, len(data)):
        current_lookback = data['lookback_'+code].iloc[i]
        
        # 根据当天的动态 lookback 截取对应长度的历史收盘价
        prices = data[code].iloc[i - current_lookback + 1 : i + 1].values
        scores[i] = calculate_score_dynamic(prices)
        
    data['得分_'+code] = scores

# ================= 5. 信号生成与净值计算 =================
# 去掉计算过程中的缺失值（前60天）
data = data.dropna()

# 取出每日得分最高的证券
data['信号'] = data[['得分_'+v for v in code_list]].idxmax(axis=1).str.replace('得分_', '')
# 今日的涨幅由昨日的持仓产生
data['信号'] = data['信号'].shift(1)
data = data.dropna()

data['轮动策略日收益率'] = data.apply(lambda x: x['日收益率_'+x['信号']], axis=1) 
# 第一天尾盘交易，当日涨幅不纳入
data.loc[data.index[0], '轮动策略日收益率'] = 0.0

# 计算累计净值
data['轮动策略净值'] = (1.0 + data['轮动策略日收益率']).cumprod()

# 查看结果
data[['得分_'+v for v in code_list] + ['信号', '轮动策略日收益率', '轮动策略净值']].head(10)

In [ ]:
# # 取出每日涨幅最大的证券
# data['信号'] = data[['涨幅_'+v for v in code_list]].idxmax(axis=1).str.replace('涨幅_', '')
# # 今日的涨幅由昨日的持仓产生
# data['信号'] = data['信号'].shift(1)
# data = data.dropna()
# data['轮动策略日收益率'] = data.apply(lambda x: x['日收益率_'+x['信号']], axis=1) 
# # 第一天尾盘交易，当日涨幅不纳入
# data.loc[data.index[0],'轮动策略日收益率'] = 0.0
# data['轮动策略净值'] = (1.0 + data['轮动策略日收益率']).cumprod()
# data[['涨幅_'+v for v in code_list]+['信号','轮动策略日收益率','轮动策略净值']].head(10)

# 取出每日得分最高的证券
data['信号'] = data[['得分_'+v for v in code_list]].idxmax(axis=1).str.replace('得分_', '')
# 今日的涨幅由昨日的持仓产生
data['信号'] = data['信号'].shift(1)
data = data.dropna()
data['轮动策略日收益率'] = data.apply(lambda x: x['日收益率_'+x['信号']], axis=1) 
# 第一天尾盘交易，当日涨幅不纳入
data.loc[data.index[0],'轮动策略日收益率'] = 0.0
data['轮动策略净值'] = (1.0 + data['轮动策略日收益率']).cumprod()
data[['得分_'+v for v in code_list]+['信号','轮动策略日收益率','轮动策略净值']].head(10)

In [ ]:
# 显示中文设置
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False

# 绘制净值曲线图
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlabel('日期')
ax.set_ylabel('净值')
name_list = []
for code in code_list:
    if code not in code_to_name_dict.keys():
        continue
    name = code_to_name_dict[code]
    name_list.append(name)
    data[name+'净值'] = data[code] / data[code].iloc[0]
    ax.plot(data.index, data[name+'净值'].values, linestyle='--')
ax.semilogy(data.index, data['轮动策略净值'].values, linestyle='-', color='#FF8124')
name_list.append('轮动策略')

# 显示图例和标题
ax.legend(name_list)
ax.set_title('轮动策略净值曲线对比')

plt.show()

In [ ]:
#将完整回测报告存为HTML文件
title = '轮动策略回测报告_ATR'
output_file = os.path.join(os.getcwd(), f'{title}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html')
qs.reports.html(data['轮动策略净值'], benchmark=data['510300'],
                title=title, output=output_file) 
print(f'已将回测报告保存到文件: {output_file}')

#输出基本回测报告信息
qs.reports.basic(data['轮动策略净值'], benchmark=data['510300']) 